# Transformer 100M CUDA — Google Colab SFT Training

**Sebelum mulai**, pastikan struktur Google Drive seperti ini:
```
MyDrive/
  transformer_100m_cuda.zip   ← source code
  sft_data/
    sft_clean/...             ← 9 file JSONL SFT
  checkpoints_100m/
    ckpt_pretrain_final.safetensors   ← pretrained checkpoint
```

> **Urutan eksekusi:** Jalankan cell satu per satu dari atas ke bawah.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — Sesuaikan path di sini

In [ ]:
import os

# ╔══════════════════════════════════════════════════════════╗
# ║  SESUAIKAN PATH INI DENGAN LOKASI FILE DI GOOGLE DRIVE  ║
# ╚══════════════════════════════════════════════════════════╝
DRIVE_ROOT = "/content/drive/MyDrive"

ZIP_PATH   = f"{DRIVE_ROOT}/transformer_100m_cuda.zip"
SFT_PATH   = f"{DRIVE_ROOT}/sft_data"

# Pretrained checkpoint — upload ke Drive dulu
BASE_MODEL = f"{DRIVE_ROOT}/checkpoints_100m/ckpt_pretrain_final.safetensors"

# Working directory di Colab (lokal — lebih cepat dari Drive)
WORK_DIR   = "/content/transformer_100m_cuda"

# Checkpoint SFT disimpan ke Drive agar persist
CKPT_DRIVE = f"{DRIVE_ROOT}/checkpoints_sft"

# ── Export sebagai env vars agar bash cell bisa akses langsung ──
os.environ["WORK_DIR"]   = WORK_DIR
os.environ["ZIP_PATH"]   = ZIP_PATH
os.environ["SFT_PATH"]   = SFT_PATH
os.environ["BASE_MODEL"] = BASE_MODEL
os.environ["CKPT_DRIVE"] = CKPT_DRIVE

# ── Validasi file ada di Drive ──
errors = []
if not os.path.exists(ZIP_PATH):
    errors.append(f"  ✗ ZIP tidak ditemukan : {ZIP_PATH}")
if not os.path.isdir(SFT_PATH):
    errors.append(f"  ✗ sft_data/ tidak ditemukan: {SFT_PATH}")
if not os.path.exists(BASE_MODEL):
    errors.append(f"  ✗ Pretrained ckpt tidak ditemukan: {BASE_MODEL}")

if errors:
    print("❌ PATH SALAH — periksa kembali di atas:")
    for e in errors: print(e)
else:
    print("✓ Semua path ditemukan")
    print(f"  ZIP        : {ZIP_PATH}")
    print(f"  sft_data/  : {SFT_PATH}")
    print(f"  base_model : {BASE_MODEL}")
    print(f"  ckpt out   : {CKPT_DRIVE}")


## 3. Extract source code ke local disk

In [ ]:
import os, shutil

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print(f"Cleaned old {WORK_DIR}")

os.makedirs(WORK_DIR, exist_ok=True)
!unzip -q "$ZIP_PATH" -d "$WORK_DIR"

assert os.path.exists(f"{WORK_DIR}/CMakeLists.txt"), \
    f"Extraction failed! Files in {WORK_DIR}:\n" + str(os.listdir(WORK_DIR)[:10])
print(f"✓ Source extracted to {WORK_DIR}")


## 4. Symlink data & pretrained checkpoint

In [ ]:
import os, shutil

def make_symlink(target, link):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    elif os.path.exists(link):
        os.remove(link)
    os.symlink(target, link)
    print(f"✓ {os.path.basename(link)}/ -> {target}")

# Symlink SFT data dari Drive
make_symlink(SFT_PATH, f"{WORK_DIR}/sft_data")

# Symlink checkpoint dir ke Drive
os.makedirs(CKPT_DRIVE, exist_ok=True)
make_symlink(CKPT_DRIVE, f"{WORK_DIR}/checkpoints")

# Copy pretrained checkpoint ke local (symlink file .safetensors kadang gagal di Colab)
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
if not os.path.exists(f"{WORK_DIR}/checkpoints/ckpt_pretrain_final.safetensors"):
    print("Copying pretrained checkpoint...")
    shutil.copy2(BASE_MODEL, f"{WORK_DIR}/checkpoints/ckpt_pretrain_final.safetensors")
    print("✓ Pretrained checkpoint copied")
else:
    print("✓ Pretrained checkpoint already exists")


## 5. Check GPU & CUDA version

In [ ]:
!nvidia-smi
!nvcc --version

## 6. Build the project

In [ ]:
%%bash
set -e
WORK_DIR="/content/transformer_100m_cuda"
cd "$WORK_DIR"

if ! command -v g++ &> /dev/null; then
    echo "Installing build-essential..."
    apt-get update -qq && apt-get install -y -qq build-essential cmake > /dev/null 2>&1
fi

export PATH="/usr/local/cuda/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:$LD_LIBRARY_PATH"

# Clean old build cache (ZIP mungkin berisi build/ dari mesin lokal)
rm -rf build

GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)
echo "Detected GPU: $GPU_NAME"

if echo "$GPU_NAME" | grep -qi "T4"; then
    GPU_TARGET="T4"
elif echo "$GPU_NAME" | grep -qi "V100"; then
    GPU_TARGET="V100"
elif echo "$GPU_NAME" | grep -qi "A100"; then
    GPU_TARGET="A100"
elif echo "$GPU_NAME" | grep -qi "L4"; then
    GPU_TARGET="RTX4090"
elif echo "$GPU_NAME" | grep -qi "H100"; then
    GPU_TARGET="H100"
else
    GPU_TARGET="ALL"
fi

echo "Building for: $GPU_TARGET"
echo "g++: $(which g++)"
echo "nvcc: $(which nvcc)"
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DGPU_TARGET=$GPU_TARGET
make -j$(nproc)
echo ""
echo "Build complete"
ls -lh train_sft_cuda 2>/dev/null || true


## 7. Start SFT Training

**Setting rekomendasi** (berdasarkan analisis spike sebelumnya):
- `lr=3e-6` — rendah karena model sudah kenal semua token (format plain text)
- `grad_clip=1.0` — mencegah grad norm spike
- `epochs=1` — 1 epoch cukup untuk SFT
- `warmup=0` — model sudah converge, tidak perlu warmup

**Batch size guide:**
- T4 (16GB)  : 12-16
- L4 (24GB)  : 20-24
- A100 (40GB): 40-48

In [ ]:
import subprocess, os

WORK_DIR = "/content/transformer_100m_cuda"

# ═══ SFT TRAINING PARAMETERS ═══
# Sesuaikan batch_size berdasarkan GPU:
#   T4 (16GB)   : 12-16
#   L4 (24GB)   : 20-24
#   A100 (40GB) : 40-48

BATCH_SIZE = 16

env = os.environ.copy()
env.pop('CUDA_VISIBLE_DEVICES', None)

os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)

print("▶ Starting SFT training (single GPU)...")
print(f"  base_model : checkpoints/ckpt_pretrain_final.safetensors")
print(f"  sft_dir    : sft_data/sft_clean")
print(f"  batch_size : {BATCH_SIZE}")
print(f"  lr         : 3e-6")
print(f"  grad_clip  : 1.0")
print(f"  epochs     : 1")
print(f"  warmup     : 0")
print("=" * 60)

proc = subprocess.Popen(
    ['stdbuf', '-oL', '-eL',
     './build/train_sft_cuda',
     '--stage', '2',
     '--base_model', 'checkpoints/ckpt_pretrain_final.safetensors',
     '--sft_dir', 'sft_data/sft_clean',
     '--vocab_dir', 'vocab_6',
     '--bpe', 'bpe_v6.bin',
     '--checkpoint_dir', 'checkpoints',
     '--batch_size', str(BATCH_SIZE),
     '--max_seq', '512',
     '--epochs', '1',
     '--lr', '3e-6',
     '--warmup', '0',
     '--grad_clip', '1.0',
     '--log_every', '5',
     '--save_every', '500'],
    cwd=WORK_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\n⚠ Training dihentikan manual (Ctrl+C).")

proc.wait()
if proc.returncode == 0:
    print("\n✓ SFT Training selesai.")
elif proc.returncode == 2:
    print("\n⚠ Training stopped: CUDA OOM. Turunkan batch_size lalu retry.")
else:
    print(f"\n✗ Training exit code: {proc.returncode}")


## 8. (Optional) Multi-GPU DDP — 2 GPU

Colab Pro+ kadang dapat 2 GPU. Jika ya, gunakan cell ini sebagai pengganti cell 7:

In [ ]:
import subprocess, sys, os

WORK_DIR = "/content/transformer_100m_cuda"
NUM_GPUS = int(subprocess.check_output("nvidia-smi -L | wc -l", shell=True).strip())
print(f"Available GPUs: {NUM_GPUS}")

if NUM_GPUS >= 2:
    print("Multi-GPU detected! Rebuilding with DDP...")

    os.system("apt-get install -y libnccl2 libnccl-dev 2>/dev/null")

    build_cmd = (
        f"rm -rf {WORK_DIR}/build && mkdir -p {WORK_DIR}/build && "
        f"cd {WORK_DIR}/build && "
        f"cmake .. -DCMAKE_BUILD_TYPE=Release -DGPU_TARGET=T4 -DUSE_DDP=ON && "
        f"make -j$(nproc)"
    )
    ret = os.system(build_cmd)
    if ret != 0:
        raise RuntimeError("DDP build failed!")
    print("✓ DDP build complete")

    # DDP batch_size per GPU, total effective = BATCH_SIZE * 2
    BATCH_SIZE = 16

    env = os.environ.copy()
    env.pop('CUDA_VISIBLE_DEVICES', None)
    env['NCCL_P2P_DISABLE'] = '1'
    env['PYTHONUNBUFFERED'] = '1'

    cmd = [
        'stdbuf', '-oL', '-eL',
        f'{WORK_DIR}/build/train_sft_cuda',
        '--stage', '2',
        '--base_model', 'checkpoints/ckpt_pretrain_final.safetensors',
        '--sft_dir', 'sft_data/sft_clean',
        '--vocab_dir', 'vocab_6',
        '--bpe', 'bpe_v6.bin',
        '--checkpoint_dir', 'checkpoints',
        '--batch_size', str(BATCH_SIZE),
        '--max_seq', '512',
        '--epochs', '1',
        '--lr', '3e-6',
        '--warmup', '0',
        '--grad_clip', '1.0',
        '--log_every', '5',
        '--save_every', '500',
        '--ddp', '--num_gpus', '2',
    ]

    print("=" * 60)
    print(f"▶ Starting SFT DDP training (2 GPUs, batch={BATCH_SIZE} each)")
    print("=" * 60)

    proc = subprocess.Popen(
        cmd,
        cwd=WORK_DIR,
        env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n⚠ Training dihentikan manual.")
    proc.wait()

else:
    print("Single GPU terdeteksi — jalankan cell 7 (tanpa DDP)")


## 9. Monitor training progress

In [ ]:
import glob

WORK_DIR = "/content/transformer_100m_cuda"

ckpts = sorted(glob.glob(f"{WORK_DIR}/checkpoints/ckpt_sft_*.safetensors"))
if ckpts:
    print(f"Latest checkpoint: {ckpts[-1]}")
    print(f"Total checkpoints: {len(ckpts)}")
else:
    print("No checkpoints yet")

!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv